# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [1]:
%pip install -q feedparser trafilatura google-generativeai python-dotenv edge-tts nest_asyncio openai-whisper moviepy==1.0.3 Pillow

Note: you may need to restart the kernel to use updated packages.


In [38]:
%pip install python-dateutil

Note: you may need to restart the kernel to use updated packages.


## Paso 1: Obtener noticias

In [40]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=1, periodo='dia')

RECOPILANDO NOTICIAS — último/a dia

-> Leyendo RSS de todas las fuentes...
  [ABC] 1 articulos en RSS
  [ElDiario] 1 articulos en RSS
  [Europa Press] 1 articulos en RSS
  [La Vanguardia] 1 articulos en RSS
  inutos] 1 articulos en RSS
  [Antena 3] 1 articulos en RSS
  [RTVE] 1 articulos en RSS
  [El Pais] 1 articulos en RSS
  [El Mundo] 1 articulos en RSS

-> 9/9 artículos dentro del período 'dia'

-> Extrayendo texto completo (9 articulos)...
  Procesando: Roba las claves de la banca online de un conocido para ...
    -> Texto completo (1247 chars)
  Procesando: Sánchez pide "no doblegarse" ante Estados Unidos para c...
    -> Texto completo (2089 chars)
  Procesando: La OCDE recorta una décima la previsión de España para ...
    -> Texto completo (8070 chars)
  Procesando: Primeras señales de alarma ante la falta de suministro ...
    -> Texto completo (5533 chars)
  Procesando: Tres de cada cuatro españoles rechaza la intervención d...
    -> Texto completo (4178 chars)
  Procesan

In [41]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ABC
Título:  Roba las claves de la banca online de un conocido para estafarle más de 18.000 euros
Origen:  completo
Chars:   1247
Preview: Valencia
Agentes de la Policía Nacional han detenido en la localidad valenciana de Torrent a un joven de 25 años como presunto autor de un delito de estafa, tras haber realizado varias transferencias ...

Artículo 2
Fuente:  ElDiario
Título:  Sánchez pide "no doblegarse" ante Estados Unidos para construir una Europa "más autónoma y más libre"
Origen:  completo
Chars:   2089
Preview: Sánchez pide “no doblegarse” ante Estados Unidos para construir una Europa “más autónoma y más libre”
4
Pedro Sánchez llama a Europa a “doblegarse” ante los Estados Unidos de Donald Trump para constru...

Artículo 3
Fuente:  Europa Press
Título:  La OCDE recorta una décima la previsión de España para 2026, hasta el 2,1%, ante la guerra de Irán
Origen:  completo
Chars:   8070
Preview: Archivo - FILED - 27 May 2015, Berlin: The logo of the Organization

## Paso 2: Generación del resumen

In [43]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
#GEMINI_API_KEY = ""

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 9 ARTÍCULOS

-> Agrupando artículos por temas...
   Error agrupando temas: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 8.358411443s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 8
}
]
Error agrupando, abortan

TypeError: cannot unpack non-iterable NoneType object

## Paso 3: Síntesis de voz (T2S)

In [ ]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       3741
  Audio:       audios/audio_2026-03-26_10-26-35.mp3
Audio generado! (1.29 MB)
  Generando subtítulos con Whisper...
  Subtítulos generados: audios/subtitulos_2026-03-26_10-26-35.vtt


In [ ]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 9195 chars
WEBVTT

00:00:00.000 --> 00:00:01.540
Muy buenas tardes, y

00:00:01.540 --> 00:00:02.560
bienvenidos a su cita

00:00:02.560 --> 00:00:04.240
con la información, bienvenidos

00:00:04.240 --> 00:00:04.740
al día.

00:00:05.460 --> 00:00:06.580
En el ámbito del

00:00:06.580 --> 00:00:08.099
fútbol 


In [ ]:
# Paso 3b — Regenerar subtítulos con el nuevo formato (2 líneas)
import importlib
import t2s
importlib.reload(t2s)

from t2s import generar_subtitulos_whisper

ruta_subtitulos = generar_subtitulos_whisper(
    ruta_audio,
    ruta_audio.replace("audio_", "subtitulos_").replace(".mp3", ".vtt")
)


  Generando subtítulos con Whisper...
  Subtítulos generados: audios/subtitulos_2026-03-26_10-26-35.vtt
WEBVTT

00:00:00.000 --> 00:00:02.839
Muy buenas tardes, y bienvenidos
a su cita con la

00:00:02.839 --> 00:00:07.259
información, bienvenidos al día. En
el ámbito del fútbol español,

00:00:07.620 --> 00:00:10.039
el Córdoba CFC se enfrenta
a una racha de seis

00:00:10.039 --> 00:00:14.000
derrotas consecutivas, incluyendo un resultado
de cuatro acero ante el

00:00:14.000 --> 00:00:17.019
burb


## Paso 4: Generación del video resumen 

In [ ]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')
#PEXELS_API_KEY = ""

ruta_video = generar_video(
    ruta_audio, ruta_subtitulos, resumenes,
    PEXELS_API_KEY,
    modelo_ia=mi_modelo_gemini  # para mejorar las queries de imagen
)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    226.0s
   Temas:             4
   Duración por tema: 56.5s

2. Buscando imágenes en Pexels...
  Tema: Fútbol y Deportes...
    Keywords: 'fútbol deportes'
    Imagen guardada: imagenes/img_4690631784009599308.jpg
  Tema: Política Española...
    Keywords: 'política española'
    Imagen guardada: imagenes/img_6428898181280191229.jpg
  Tema: Conflicto y Política Internacional (Irán)...
    Keywords: 'conflicto política internacional (irán)'
    Imagen guardada: imagenes/img_3423083709370468111.jpg
  Tema: Temas Sociales y Éticos...
    Keywords: 'temas sociales éticos'
    Imagen guardada: imagenes/img_4205699815993719858.jpg

3. Creando clips...
   [1/4] Fútbol y Deportes...
   [2/4] Política Española...
   [3/4] Conflicto y Política Internacional (Irán...
   [4/4] Temas Sociales y Éticos...

4. Concatenando clips y añadiendo audio...

5. Exportando vídeo base...

5b. Escalando a resolución completa...

6. Añadiendo subtítulos